In [ ]:
# Cell 1: Imports
import os
import time
import tempfile
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_add_pool, GraphNorm
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.utils import softmax, add_self_loops
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_curve, auc,
    f1_score, matthews_corrcoef, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from rdkit import Chem
from rdkit.Chem import AllChem

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# Cell 2: Feature helpers and mol_to_graph (ONE-HOT encoding)

# One-hot helper
def one_hot(idx, size):
    vec = [0.0] * size
    if idx < 0:
        idx = 0
    if idx >= size:
        idx = size - 1
    vec[idx] = 1.0
    return vec

# Caps for one-hot sizes (adjust if you expect exotic atoms)
MAX_ATOMIC_NUM = 100   
MAX_DEGREE = 6        
MAX_HYBRID = 6         
MAX_TOTAL_H = 5       
MAX_BOND_TYPE = 5     
MAX_BOND_STEREO = 6 

def atom_features_onehot(atom):
    # atomic number -> one-hot
    an = atom.GetAtomicNum()
    deg = atom.GetDegree()
    charge = atom.GetFormalCharge()
    hyb = int(atom.GetHybridization()) if atom.GetHybridization() is not None else 0
    aromatic = int(atom.GetIsAromatic())
    total_h = atom.GetTotalNumHs()
    chirality = int(atom.HasProp('_ChiralityPossible'))

    feats = []
    feats += one_hot(min(an, MAX_ATOMIC_NUM-1), MAX_ATOMIC_NUM)   
    feats += one_hot(min(deg, MAX_DEGREE-1), MAX_DEGREE)
    feats.append(float(charge))
    feats += one_hot(min(hyb, MAX_HYBRID-1), MAX_HYBRID)
    feats.append(float(aromatic))
    feats += one_hot(min(total_h, MAX_TOTAL_H-1), MAX_TOTAL_H)
    feats.append(float(chirality))
    return feats

def bond_features_onehot(bond):
    try:
        btype = int(bond.GetBondTypeAsDouble())
    except Exception:
        btype = 0
    conj = int(bond.GetIsConjugated())
    ring = int(bond.IsInRing())
    stereo = int(bond.GetStereo())

    feats = []
    feats += one_hot(min(btype, MAX_BOND_TYPE-1), MAX_BOND_TYPE)
    feats.append(float(conj))
    feats.append(float(ring))
    feats += one_hot(min(stereo, MAX_BOND_STEREO-1), MAX_BOND_STEREO)
    return feats

def mol_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        Chem.Kekulize(mol, clearAromaticFlags=True)
    except Exception:
        pass

    atom_feats = [atom_features_onehot(atom) for atom in mol.GetAtoms()]
    if len(atom_feats) == 0:
        return None
    x = torch.tensor(atom_feats, dtype=torch.float)

    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        bf = bond_features_onehot(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr += [bf, bf]

    BOND_FEATURE_SIZE = len(one_hot(0, MAX_BOND_TYPE) + [0.0, 0.0] + one_hot(0, MAX_BOND_STEREO))

    if len(edge_index) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, BOND_FEATURE_SIZE), dtype=torch.float) 
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)


    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)


In [ ]:
# Cell 3: Dataset wrapper
class SMILESDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_excel(file_path)
        self.graphs = []
        self.labels = []
        for _, row in df.iterrows():
            data = mol_to_graph(str(row['Smiles']))
            if data is None:
                continue
            data.y = torch.tensor(float(row['Labels']), dtype=torch.float)
            self.graphs.append(data)
            self.labels.append(int(row['Labels']))

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, idx):
        return self.graphs[idx]

    def get_labels(self):
        return self.labels

In [ ]:
# Cell 4: AAGINConvSingle and MultiHead wrapper
class AAGINConvSingle(MessagePassing):
    def __init__(self, in_dim, out_dim, edge_dim, eps_init=0.0, dropout=0.0):
        super().__init__(aggr='add', node_dim=0)
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.edge_dim = edge_dim
        self.dropout = dropout

        # learnable eps parameter
        self.eps = nn.Parameter(torch.tensor([eps_init], dtype=torch.float))

        # projections
        self.lin_node = nn.Linear(in_dim, out_dim, bias=True)
        self.lin_edge = nn.Linear(edge_dim, out_dim, bias=True)

        self.msg_mlp = nn.Sequential(
            nn.Linear(out_dim + out_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )
        self.att_mlp = nn.Sequential(
            nn.Linear(out_dim * 3, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, 1)
        )

        self.update_mlp = nn.Sequential(
            nn.Linear(out_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )

    def forward(self, x, edge_index, edge_attr):
        if edge_index.numel() == 0:
            edge_index = torch.empty((2, 0), dtype=torch.long, device=x.device)

        try:
            edge_index, edge_attr = add_self_loops(edge_index, edge_attr=edge_attr, num_nodes=x.size(0))
        except TypeError:
            edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
            if edge_attr is None:
                edge_attr = torch.zeros((edge_index.size(1), self.edge_dim), dtype=x.dtype, device=x.device)
            elif edge_attr.size(0) < edge_index.size(1):
                pad = torch.zeros((edge_index.size(1) - edge_attr.size(0), self.edge_dim), dtype=x.dtype, device=x.device)
                edge_attr = torch.cat([edge_attr.to(x.device), pad], dim=0)
        else:
            if edge_attr is None:
                edge_attr = torch.zeros((edge_index.size(1), self.edge_dim), dtype=x.dtype, device=x.device)
            else:
                edge_attr = edge_attr.to(x.device)

        x_proj = self.lin_node(x)
        if self.edge_dim > 0:
            e_proj = self.lin_edge(edge_attr)
        else:
            e_proj = self.lin_edge(torch.zeros((edge_index.size(1),1), dtype=x.dtype, device=x.device))

        return self.propagate(edge_index, x=x_proj, e=e_proj)

    def message(self, x_i, x_j, e, index):
        msg_in = torch.cat([x_j, e], dim=-1)
        m = self.msg_mlp(msg_in)
        att_in = torch.cat([x_i, x_j, e], dim=-1)
        logits = self.att_mlp(att_in).view(-1)
        alpha = softmax(logits, index)
        if self.dropout > 0 and self.training:
            alpha = F.dropout(alpha, p=self.dropout)
        return m * alpha.view(-1, 1)

    def update(self, aggr_out, x):
        out = (1.0 + self.eps) * x + aggr_out
        out = self.update_mlp(out)
        return out

class MultiHeadAAGINConv(nn.Module):
    def __init__(self, in_dim, head_out_dim, heads, edge_dim, concat=True, eps_init=0.0, dropout=0.0):
        super().__init__()
        self.heads = heads
        self.concat = concat
        self.heads_list = nn.ModuleList([AAGINConvSingle(in_dim, head_out_dim, edge_dim=edge_dim, eps_init=eps_init, dropout=dropout) for _ in range(heads)])

    def forward(self, x, edge_index, edge_attr):
        head_outs = [head(x, edge_index, edge_attr) for head in self.heads_list]
        if self.concat:
            return torch.cat(head_outs, dim=-1)
        else:
            stacked = torch.stack(head_outs, dim=0)
            return torch.mean(stacked, dim=0)


In [ ]:
# Cell 5: AAGINNetMultiHead
class AAGINNetMultiHead(nn.Module):
    def __init__(self, input_dim, edge_dim, hidden_dim=64, num_layers=3,
                 num_heads=4, concat=True, activation="relu", dropout=0.2, use_residual=False):
        super().__init__()
        assert num_layers >= 1, "Use at least 1 GNN layer"
        self.act = getattr(F, activation)
        self.dropout = dropout
        self.use_residual = use_residual
        self.concat = concat
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.hidden_dim = hidden_dim

        # first_in matches what each MultiHead expects as input
        first_in = hidden_dim * num_heads if concat else hidden_dim

        # project raw atom features -> first_in
        self.input_proj = nn.Linear(input_dim, first_in)

        # build multiple MultiHeadAAGINConv layers
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()

        self.res_projs = nn.ModuleList()

        cur_in = first_in
        for i in range(num_layers):
            layer = MultiHeadAAGINConv(in_dim=cur_in, head_out_dim=hidden_dim, heads=num_heads,
                                       edge_dim=edge_dim, concat=concat, dropout=dropout)
            self.layers.append(layer)

            out_dim = (hidden_dim * num_heads) if concat else hidden_dim
            self.norms.append(GraphNorm(out_dim))

            if self.use_residual:
                if cur_in == out_dim:
                    self.res_projs.append(nn.Identity())
                else:
                    self.res_projs.append(nn.Linear(cur_in, out_dim))
            else:
                self.res_projs.append(nn.Identity())
  
            cur_in = out_dim

        # final MLP head
        self.fc1 = nn.Linear(cur_in, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.input_proj(x)

        for idx, (layer, norm, res_proj) in enumerate(zip(self.layers, self.norms, self.res_projs)):
            x_in = x 

            x_out = layer(x, edge_index, edge_attr)

            if self.use_residual:
                x_skip = res_proj(x_in)
                x = x_out + x_skip
            else:
                x = x_out

            # normalization + activation + dropout
            x = norm(x)
            x = self.act(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        # graph pooling + final MLP
        x_pool = global_add_pool(x, batch)
        x = self.fc1(x_pool)
        x = self.act(x)
        x = self.fc2(x)
        return x.view(-1)

    # Feature extraction
    def extract_features(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch

        x = self.input_proj(x)
        for idx, (layer, norm, res_proj) in enumerate(zip(self.layers, self.norms, self.res_projs)):
            x_in = x
            x_out = layer(x, edge_index, edge_attr)
            if self.use_residual:
                x_skip = res_proj(x_in)
                x = x_out + x_skip
            else:
                x = x_out
            x = norm(x)
            x = self.act(x)
            x = F.dropout(x, p=self.dropout, training=self.training) 
        
        # --- Graph pooling ---
        x_pool = global_add_pool(x, batch)
        
        # Return the pooled graph-level features
        return x_pool

In [ ]:
# Cell 6: Robust metrics 
def calculate_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.asarray(y_prob).astype(float).ravel()

    if len(np.unique(y_true)) > 1:
        try:
            auc_score = roc_auc_score(y_true, y_prob)
        except Exception:
            auc_score = float('nan')
        try:
            precision, recall, _ = precision_recall_curve(y_true, y_prob)
            auprc = auc(recall, precision)
        except Exception:
            auprc = float('nan')
    else:
        auc_score = float('nan')
        auprc = float('nan')

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.0
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        'AUC': float(auc_score),
        'AUPRC': float(auprc),
        'Accuracy': float(acc),
        'F1': float(f1),
        'MCC': float(mcc),
        'Sensitivity': float(sensitivity),
        'Specificity': float(specificity)
    }

def make_pos_weight_from_labels(labels):
    classes = np.array([0, 1])
    cw = compute_class_weight(class_weight='balanced', classes=classes, y=np.array(labels))
    return torch.tensor(cw[1], dtype=torch.float, device=device)


In [ ]:
# Cell 7: Training loop 
def train_model(model, train_loader, val_loader, optimizer, pos_weight,
                patience=8, max_epochs=200, checkpoint_path=None,
                use_amp=True, clip_norm=2.0):
    use_amp = use_amp and torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler() if use_amp else None

    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=False
    )
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    start_time = time.time()

    for epoch in range(max_epochs):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            if use_amp:
                with torch.cuda.amp.autocast():
                    out = model(batch)
                    loss = criterion(out, batch.y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(batch)
                loss = criterion(out, batch.y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_norm)
                optimizer.step()

        # Validation
        model.eval()
        val_loss = 0.0
        n_val = 0
        with torch.no_grad():
            for vb in val_loader:
                vb = vb.to(device)
                if use_amp:
                    with torch.cuda.amp.autocast():
                        out = model(vb)
                        loss = criterion(out, vb.y)
                else:
                    out = model(vb)
                    loss = criterion(out, vb.y)
                val_loss += loss.item()
                n_val += 1
        val_loss = val_loss / max(1, n_val)

        scheduler.step(val_loss)

        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            best_model_state = deepcopy(model.state_dict())
            if checkpoint_path:
                tmpf = checkpoint_path + ".tmp"
                torch.save(best_model_state, tmpf)
                os.replace(tmpf, checkpoint_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    training_time = time.time() - start_time
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model, training_time

# Helper function to extract embeddings and labels
@torch.no_grad() 
def get_embeddings_and_labels(loader, model, device):
    """
    Runs a data loader through the model's feature extractor.
    Returns:
        X: numpy array of feature embeddings
        y: numpy array of corresponding labels
    """
    model.eval() 
    all_features = []
    all_labels = []
    
    for data in loader:
        data = data.to(device)
        
        features = model.extract_features(data) 
        
        all_features.append(features.cpu().numpy())
        all_labels.append(data.y.cpu().numpy()) 
        
    X = np.concatenate(all_features, axis=0)
    y = np.concatenate(all_labels, axis=0).astype(int) 
    return X, y

In [ ]:
# Cell 8: Hyperparameters 

FIXED_HYPERPARAMS = {
    "lr": 0.0005,
    "dropout": 0.1,
    "optimizer": "AdamW",
    "weight_decay": 0.0001,
    "batch_size": 64,
    "hidden_dim": 64,
    "num_layers": 3,
    "activation": "elu",
    "num_heads": 8,
    "concat": False
}

def build_optimizer(name, params, lr, wd):
    if name == "Adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=wd)
    if name == "AdamW":
        return torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    if name == "RAdam":
        return torch.optim.RAdam(params, lr=lr, weight_decay=wd)
    raise ValueError(f"Unknown optimizer: {name}")

In [ ]:
# Cell 9: Stratified 10-fold CV (AAGIN + RF)
def stratified_kfold_cv(dataset, n_splits=10, seed=42, hyperparams=FIXED_HYPERPARAMS):
    labels = np.array(dataset.get_labels())
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    all_fold_metrics = []
    fold_times = []
    ckpt_paths = [] 

    input_dim = dataset[0].x.shape[1]
    edge_attr_sample = dataset[0].edge_attr
    edge_dim = edge_attr_sample.shape[1] if (edge_attr_sample is not None and edge_attr_sample.numel() > 0) else 0

    for fold, (train_idx, test_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        print(f"\n=== Outer fold {fold+1}/{n_splits} ===")
        # -----------------------------------------------------------------
        # STAGE 1: Train GNN Feature Extractor for this Fold 
        # -----------------------------------------------------------------
        tr_labels = labels[train_idx]
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=seed + fold)
        tr_sub_idx, val_sub_idx = next(sss.split(np.zeros(len(tr_labels)), tr_labels))
        tr_final_idx = np.array(train_idx)[tr_sub_idx]
        val_final_idx = np.array(train_idx)[val_sub_idx]

        train_subset = [dataset[i] for i in tr_final_idx]
        val_subset = [dataset[i] for i in val_final_idx]
        
        bs = hyperparams["batch_size"]
        train_loader = DataLoader(train_subset, batch_size=bs, shuffle=True, drop_last=False)
        val_loader = DataLoader(val_subset, batch_size=bs, shuffle=False)

        model = AAGINNetMultiHead(
            input_dim=input_dim,
            edge_dim=edge_dim,
            hidden_dim=hyperparams["hidden_dim"],
            num_layers=hyperparams["num_layers"],
            num_heads=hyperparams["num_heads"],
            concat=hyperparams["concat"],
            activation=hyperparams["activation"],
            dropout=hyperparams["dropout"],
            use_residual=False,
        ).to(device)

        optimizer = build_optimizer(hyperparams["optimizer"], model.parameters(), hyperparams["lr"], hyperparams["weight_decay"])
        pos_w = make_pos_weight_from_labels([dataset.get_labels()[i] for i in tr_final_idx])

        ckpt_tmp = tempfile.NamedTemporaryFile(delete=False, suffix=f"_fold{fold+1}.pth")
        ckpt_tmp.close()
        ckpt_path = ckpt_tmp.name
        try:
            start = time.time()
            model, _ = train_model(model, train_loader, val_loader, optimizer,
                                   pos_weight=pos_w,
                                   patience=8, max_epochs=200,
                                   checkpoint_path=ckpt_path,
                                   use_amp=True, clip_norm=2.0)
            fold_times.append(time.time() - start) 
            ckpt_paths.append(ckpt_path)

            # -----------------------------------------------------------------
            # STAGE 2: Train & Evaluate Random Forest on GNN Features 
            # -----------------------------------------------------------------
            print(f"Fold {fold+1}: GNN training complete. Extracting features for RF...")
            
            rf_train_subset = [dataset[i] for i in train_idx]
            rf_test_subset = [dataset[i] for i in test_idx]
            
            rf_train_loader = DataLoader(rf_train_subset, batch_size=bs, shuffle=False)
            rf_test_loader = DataLoader(rf_test_subset, batch_size=bs, shuffle=False)
            
            X_train_rf, y_train_rf = get_embeddings_and_labels(rf_train_loader, model, device)
            X_test_rf, y_test_rf = get_embeddings_and_labels(rf_test_loader, model, device)

            print(f"Fold {fold+1} Train embedding shape: {X_train_rf.shape}")
            print(f"Fold {fold+1} Test embedding shape: {X_test_rf.shape}")

            print(f"Fold {fold+1}: Features extracted. Training Random Forest...")
            rf_classifier = RandomForestClassifier(
                n_estimators=200, 
                random_state=seed + fold,
                n_jobs=-1
            )
            
            rf_classifier.fit(X_train_rf, y_train_rf)
            
            print(f"Fold {fold+1}: Evaluating Random Forest...")
            y_pred_rf = rf_classifier.predict(X_test_rf)
            y_prob_rf = rf_classifier.predict_proba(X_test_rf)[:, 1] 

            metrics = calculate_metrics(y_test_rf, y_pred_rf, y_prob_rf)
            all_fold_metrics.append(metrics)
            
            print(f"Outer fold {fold+1} RF metrics:", metrics)

        except Exception as e:
            print(f"Fold {fold+1} failed with exception:", e)
            all_fold_metrics.append({k: float('nan') for k in ['AUC','AUPRC','Accuracy','F1','MCC','Sensitivity','Specificity']})
            fold_times.append(0.0)
            ckpt_paths.append(ckpt_path)
            
    return all_fold_metrics, ckpt_paths, fold_times

In [ ]:
# Cell 10: Train final (AAGINE + RF) and evaluate on test set
def train_full_from_best_fold_and_eval(train_dataset, test_dataset, fold_ckpt_paths, fold_metrics, hyperparams=FIXED_HYPERPARAMS, seed=777):
    # -----------------------------------------------------------------
    # STAGE 1: Train Final GNN Feature Extractor
    # -----------------------------------------------------------------
    aucs = [m['AUC'] if (m is not None and 'AUC' in m and np.isfinite(m['AUC'])) else -np.inf for m in fold_metrics]
    best_fold_idx = int(np.argmax(aucs))
    best_ckpt = fold_ckpt_paths[best_fold_idx]
    print(f"\nSelected best GNN fold {best_fold_idx+1} (RF AUC={aucs[best_fold_idx]:.4f}) for initializing full-train.")
    
    input_dim = train_dataset[0].x.shape[1]
    edge_attr_sample = train_dataset[0].edge_attr
    edge_dim = edge_attr_sample.shape[1] if (edge_attr_sample is not None and edge_attr_sample.numel() > 0) else 0

    model = AAGINNetMultiHead(
        input_dim=input_dim,
        edge_dim=edge_dim,
        hidden_dim=hyperparams["hidden_dim"],
        num_layers=hyperparams["num_layers"],
        num_heads=hyperparams["num_heads"],
        concat=hyperparams["concat"],
        activation=hyperparams["activation"],
        dropout=hyperparams["dropout"],
        use_residual=False,
    ).to(device)

    try:
        state = torch.load(best_ckpt, map_location=device)
        model.load_state_dict(state)
        print("Loaded best fold GNN weights into model for full training.")
    except Exception as e:
        print("Could not load best fold GNN checkpoint; starting from random init. Exception:", e)

    labels = np.array(train_dataset.get_labels())
    sss = StratifiedShuffleSplit(n_splits=1, test_size=max(1, int(0.05 * len(labels))) / len(labels), random_state=seed)
    tr_idx, val_idx = next(sss.split(np.zeros(len(labels)), labels))

    train_subset = [train_dataset[i] for i in tr_idx]
    val_subset = [train_dataset[i] for i in val_idx]
    bs = hyperparams["batch_size"]
    train_loader = DataLoader(train_subset, batch_size=bs, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_subset, batch_size=bs, shuffle=False)

    optimizer = build_optimizer(hyperparams["optimizer"], model.parameters(), hyperparams["lr"], hyperparams["weight_decay"])
    pos_w = make_pos_weight_from_labels([train_dataset.get_labels()[i] for i in tr_idx])

    ckpt_final = "best_final_GNN_model.pth"
    model, train_time_gnn = train_model(model, train_loader, val_loader, optimizer,
                                     pos_weight=pos_w,
                                     patience=8, max_epochs=200,
                                     checkpoint_path=ckpt_final,
                                     use_amp=True, clip_norm=2.0)
    
    print(f"Full GNN training complete in {train_time_gnn:.2f}s.")

    # -----------------------------------------------------------------
    # STAGE 2: Train & Evaluate Final Random Forest
    # -----------------------------------------------------------------
    print("Extracting features from full training set for final RF...")
    
    rf_train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=False)
    rf_test_loader = DataLoader(test_dataset, batch_size=bs, shuffle=False)
    
    X_train_final_rf, y_train_final_rf = get_embeddings_and_labels(rf_train_loader, model, device)
    
    print("Extracting features from independent test set...")
    start_extract_test = time.time()
    X_test_final_rf, y_test_final_rf = get_embeddings_and_labels(rf_test_loader, model, device)
    extract_test_time = time.time() - start_extract_test

    print(f"Final Train embedding shape: {X_train_final_rf.shape}")
    print(f"Final Test embedding shape: {X_test_final_rf.shape}")

    print("Training final Random Forest on full training set features...")
    rf_final_classifier = RandomForestClassifier(
        n_estimators=200, 
        random_state=seed,
        n_jobs=-1
    )
    
    start_rf_train = time.time()
    rf_final_classifier.fit(X_train_final_rf, y_train_final_rf)
    train_time_rf = time.time() - start_rf_train
    print(f"Final RF training complete in {train_time_rf:.2f}s.")

    print("Evaluating final Random Forest on test set features...")
    start_rf_test = time.time()
    y_pred_rf = rf_final_classifier.predict(X_test_final_rf)
    y_prob_rf = rf_final_classifier.predict_proba(X_test_final_rf)[:, 1]
    rf_test_time = time.time() - start_rf_test
    
    test_metrics = calculate_metrics(y_test_final_rf, y_pred_rf, y_prob_rf)

    total_train_time = train_time_gnn + train_time_rf
    total_test_time = extract_test_time + rf_test_time

    try:
        if os.path.exists(ckpt_final):
            os.remove(ckpt_final)
    except Exception:
        pass

    return {
        "best_fold_index": best_fold_idx,
        "train_time_full": total_train_time, 
        "test_metrics": test_metrics,
        "testing_time": total_test_time
    }

In [ ]:
# Cell 11: main() orchestrates 10-fold CV + final full-training (from best fold) + test eval
def main(
    train_path,
    test_path,
    random_seed=42,
    n_folds=10,
    hyperparams=FIXED_HYPERPARAMS
):

    set_global_seed(random_seed)

    train_dataset = SMILESDataset(train_path)
    test_dataset = SMILESDataset(test_path)

    # Run stratified K-fold CV
    all_fold_metrics, ckpt_paths, fold_times  = stratified_kfold_cv(train_dataset, 
                                    n_splits=n_folds, seed=random_seed, hyperparams=hyperparams)
    
    # Aggregate CV metrics
    keys = list(all_fold_metrics[0].keys())
    mean_metrics = {k: float(np.nanmean([m[k] for m in all_fold_metrics])) for k in keys}
    std_metrics = {k: float(np.nanstd([m[k] for m in all_fold_metrics])) for k in keys}

    print("\nStratified K-Fold aggregated results (mean ± std):")
    for k in mean_metrics:
        print(f"  {k}: {mean_metrics[k]:.4f} ± {std_metrics[k]:.4f}")

    # Train full model starting from best fold weights and evaluate on test set
    full_train_res = train_full_from_best_fold_and_eval(train_dataset, 
                                test_dataset, ckpt_paths, all_fold_metrics, 
                                hyperparams=hyperparams, seed=random_seed+555)

    print("\nIndependent Test Metrics:")
    for k, v in full_train_res["test_metrics"].items():
        print(f"  {k}: {v:.4f}")
    print(f"Full training time: {full_train_res['train_time_full']:.2f} s")
    print(f"Independent testing time: {full_train_res['testing_time']:.2f} s")

    for p in ckpt_paths:
        try:
            if os.path.exists(p):
                os.remove(p)
        except Exception:
            pass

    return {
        "cv_fold_metrics": all_fold_metrics,
        "cv_mean_metrics": mean_metrics,
        "cv_std_metrics": std_metrics,
        "fold_times": fold_times,
        "final_test_metrics": full_train_res["test_metrics"],
        "final_train_time": full_train_res["train_time_full"],
        "final_testing_time": full_train_res["testing_time"],
        "best_fold_index": full_train_res["best_fold_index"]
    }

In [ ]:
# Cell 12: Multi-seed evaluation + save results
import json
#from scipy.stats import wilcoxon, ttest_rel

MODEL_NAME = "AAGINE_RF"    
DATASET_NAME = "BBB"       

SEEDS = [7, 42, 77, 999, 2023]

all_seed_results = []
for sd in SEEDS:
    print(f"\n{'='*30}\nRunning full pipeline with seed = {sd}\n{'='*30}")
    res = main(
        train_path=r"BBA train_cleaned.xlsx",
        test_path=r"BBA test_cleaned.xlsx",
        random_seed=sd,
        n_folds=10,
        hyperparams=FIXED_HYPERPARAMS
    )
    res["seed"] = sd
    all_seed_results.append(res)

metric_keys = list(all_seed_results[0]["cv_mean_metrics"].keys())

cv_seed_means = {k: [r["cv_mean_metrics"][k] for r in all_seed_results] for k in metric_keys}
cv_across_seed_mean = {k: float(np.mean(v)) for k, v in cv_seed_means.items()}
cv_across_seed_std  = {k: float(np.std(v))  for k, v in cv_seed_means.items()}

test_seed_values = {k: [r["final_test_metrics"][k] for r in all_seed_results] for k in metric_keys}
test_across_seed_mean = {k: float(np.mean(v)) for k, v in test_seed_values.items()}
test_across_seed_std  = {k: float(np.std(v))  for k, v in test_seed_values.items()}

print("\n10-fold CV performance across seeds (mean ± std):")
for k in metric_keys:
    print(f"  {k}: {cv_across_seed_mean[k]:.4f} ± {cv_across_seed_std[k]:.4f}")

print("\nIndependent test-set performance across seeds (mean ± std):")
for k in metric_keys:
    print(f"  {k}: {test_across_seed_mean[k]:.4f} ± {test_across_seed_std[k]:.4f}")

out_path = f"{MODEL_NAME}_{DATASET_NAME}_multiseed_results.json"
with open(out_path, "w") as f:
    json.dump({
        "model": MODEL_NAME,
        "dataset": DATASET_NAME,
        "seeds": SEEDS,
        "per_seed_cv_fold_metrics": [r["cv_fold_metrics"] for r in all_seed_results],
        "per_seed_cv_mean_metrics": [r["cv_mean_metrics"] for r in all_seed_results],
        "per_seed_test_metrics": [r["final_test_metrics"] for r in all_seed_results],
        "cv_across_seed_mean": cv_across_seed_mean,
        "cv_across_seed_std": cv_across_seed_std,
        "test_across_seed_mean": test_across_seed_mean,
        "test_across_seed_std": test_across_seed_std,
    }, f, indent=2)

print(f"\nSaved results to {out_path}")


Running full pipeline with seed = 7


[14:04:20] WARNING: not removing hydrogen atom without neighbors
[14:04:20] WARNING: not removing hydrogen atom without neighbors
[14:04:20] WARNING: not removing hydrogen atom without neighbors
[14:04:20] WARNING: not removing hydrogen atom without neighbors
[14:04:20] WARNING: not removing hydrogen atom without neighbors
[14:04:21] WARNING: not removing hydrogen atom without neighbors
[14:04:21] WARNING: not removing hydrogen atom without neighbors
[14:04:21] WARNING: not removing hydrogen atom without neighbors
[14:04:21] WARNING: not removing hydrogen atom without neighbors
[14:04:21] WARNING: not removing hydrogen atom without neighbors
[14:04:21] WARNING: not removing hydrogen atom without neighbors
[14:04:21] WARNING: not removing hydrogen atom without neighbors
[14:04:21] WARNING: not removing hydrogen atom without neighbors



=== Outer fold 1/10 ===


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\module.py:1160: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(
C:\Users\Admin\AppData\Local\Temp\ipykernel_25820\633968843.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if use_amp else None
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
C:\Users\Admin\AppData\Local\Temp\ipykernel_25820\633968843.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp

Fold 1: GNN training complete. Extracting features for RF...
Fold 1 Train embedding shape: (2941, 64)
Fold 1 Test embedding shape: (327, 64)
Fold 1: Features extracted. Training Random Forest...
Fold 1: Evaluating Random Forest...
Outer fold 1 RF metrics: {'AUC': 0.8321095571095571, 'AUPRC': 0.8761924827615406, 'Accuracy': 0.764525993883792, 'F1': 0.8117359413202934, 'MCC': 0.5031095380894146, 'Sensitivity': 0.8512820512820513, 'Specificity': 0.6363636363636364}

=== Outer fold 2/10 ===


...

Fold 9: GNN training complete. Extracting features for RF...
Fold 9 Train embedding shape: (2942, 64)
Fold 9 Test embedding shape: (326, 64)
Fold 9: Features extracted. Training Random Forest...
Fold 9: Evaluating Random Forest...
Outer fold 9 RF metrics: {'AUC': 0.8789587003327461, 'AUPRC': 0.9035592763283504, 'Accuracy': 0.7944785276073619, 'F1': 0.8286445012787724, 'MCC': 0.5719553881370169, 'Sensitivity': 0.8307692307692308, 'Specificity': 0.7404580152671756}

=== Outer fold 10/10 ===


C:\Users\Admin\AppData\Local\Temp\ipykernel_25820\633968843.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if use_amp else None
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
C:\Users\Admin\AppData\Local\Temp\ipykernel_25820\633968843.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
C:\Users\Admin\AppData\Local\Temp\ipykernel_25820\633968843.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Fold 10: GNN training complete. Extracting features for RF...
Fold 10 Train embedding shape: (2942, 64)
Fold 10 Test embedding shape: (326, 64)
Fold 10: Features extracted. Training Random Forest...
Fold 10: Evaluating Random Forest...
Outer fold 10 RF metrics: {'AUC': 0.8542376198864748, 'AUPRC': 0.8978894760785542, 'Accuracy': 0.7852760736196319, 'F1': 0.825, 'MCC': 0.5488179661829365, 'Sensitivity': 0.8461538461538461, 'Specificity': 0.6946564885496184}

Stratified K-Fold aggregated results (mean ± std):
  AUC: 0.8494 ± 0.0265
  AUPRC: 0.8895 ± 0.0217
  Accuracy: 0.7742 ± 0.0228
  F1: 0.8164 ± 0.0174
  MCC: 0.5254 ± 0.0496
  Sensitivity: 0.8393 ± 0.0208
  Specificity: 0.6773 ± 0.0461

Selected best GNN fold 1 (RF AUC=0.8949) for initializing full-train.
Loaded best fold GNN weights into model for full training.


C:\Users\Admin\AppData\Local\Temp\ipykernel_25820\822407793.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(best_ckpt, map_location=device)
C:\Users\

Full GNN training complete in 109.77s.
Extracting features from full training set for final RF...
Extracting features from independent test set...
Final Train embedding shape: (3268, 64)
Final Test embedding shape: (817, 64)
Training final Random Forest on full training set features...
Final RF training complete in 0.57s.
Evaluating final Random Forest on test set features...

Independent Test Metrics:
  AUC: 0.8574
  AUPRC: 0.8922
  Accuracy: 0.7809
  F1: 0.8205
  MCC: 0.5404
  Sensitivity: 0.8364
  Specificity: 0.6982
Full training time: 110.35 s
Independent testing time: 0.36 s

10-fold CV performance across seeds (mean ± std):
  AUC: 0.8483 ± 0.0025
  AUPRC: 0.8889 ± 0.0017
  Accuracy: 0.7729 ± 0.0039
  F1: 0.8154 ± 0.0033
  MCC: 0.5228 ± 0.0082
  Sensitivity: 0.8392 ± 0.0052
  Specificity: 0.6743 ± 0.0059

Independent test-set performance across seeds (mean ± std):
  AUC: 0.8626 ± 0.0050
  AUPRC: 0.8953 ± 0.0045
  Accuracy: 0.7907 ± 0.0086
  F1: 0.8294 ± 0.0070
  MCC: 0.5602 ± 0.0